[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/pandera-certified/notebooks/day-07-capstone-pipeline-validation.ipynb#scrollTo=aa1122bb)

---
# Day 7 · Capstone — Production Pipeline Validation with Full Test Coverage
**certified-journeys / pandera-certified** · Three-stage pipeline · DataFrameModel · Hypothesis · pytest

> **Goal for today:** Build a fully validated three-stage data pipeline (raw ingestion → cleaning → feature engineering), guard every stage with `@pa.check_input` / `@pa.check_output`, add custom classmethods, collect lazy validation reports, and cover everything with unit + property-based tests.


In [ ]:
%pip install -q "pandera[hypotheses]" hypothesis pytest


## Step 1 · Pipeline architecture and schema design

Our pipeline processes **sensor readings** from an IoT fleet through three stages:

```
Raw ingestion           Cleaning               Feature engineering
──────────────          ────────────           ───────────────────
RawSchema        →      CleanSchema     →      FeatureSchema
ingest_raw()            clean_readings()       engineer_features()
```

| Stage | Input | Output | Key transformation |
|-------|-------|--------|--------------------|
| Ingestion | Raw CSV-like dict | `RawSchema` | Type coercion, column rename |
| Cleaning | `RawSchema` | `CleanSchema` | Drop nulls, clip outliers |
| Feature Eng. | `CleanSchema` | `FeatureSchema` | Compute ratio, z-score |

All schemas use the class-based `pa.DataFrameModel` API for cleaner IDE support.


In [ ]:
import pandas as pd
import numpy as np
import pandera as pa
from pandera import check_input, check_output
from pandera.typing import Series
from typing import Optional

# ── Stage 1: RawSchema ────────────────────────────────────────────────────
class RawSchema(pa.DataFrameModel):
    """Schema for data as it arrives from the sensor feed."""
    device_id: Series[str] = pa.Field(str_length={"min_value": 3})
    temperature: Series[float] = pa.Field(ge=-40.0, le=85.0, nullable=True)
    humidity: Series[float] = pa.Field(ge=0.0, le=100.0, nullable=True)
    timestamp: Series[str] = pa.Field(str_length={"min_value": 10})

    class Config:
        strict = False  # allow extra columns (sensor metadata)
        coerce = True   # coerce dtypes if possible


# ── Stage 2: CleanSchema ──────────────────────────────────────────────────
class CleanSchema(pa.DataFrameModel):
    """Schema after null removal and outlier clipping."""
    device_id: Series[str] = pa.Field(str_length={"min_value": 3})
    temperature: Series[float] = pa.Field(ge=-40.0, le=85.0, nullable=False)  # no nulls
    humidity: Series[float] = pa.Field(ge=0.0, le=100.0, nullable=False)
    timestamp: Series[str] = pa.Field(str_length={"min_value": 10})

    class Config:
        strict = False
        coerce = True


# ── Stage 3: FeatureSchema ────────────────────────────────────────────────
class FeatureSchema(pa.DataFrameModel):
    """Schema after feature engineering."""
    device_id: Series[str] = pa.Field(str_length={"min_value": 3})
    temperature: Series[float] = pa.Field(ge=-40.0, le=85.0, nullable=False)
    humidity: Series[float] = pa.Field(ge=0.0, le=100.0, nullable=False)
    timestamp: Series[str] = pa.Field(str_length={"min_value": 10})
    heat_index: Series[float] = pa.Field(nullable=False)
    temp_z_score: Series[float] = pa.Field(nullable=False)

    class Config:
        strict = False
        coerce = True

    @pa.check("heat_index", name="heat_index_bounded")
    @classmethod
    def heat_index_upper_bound(cls, series: Series[float]) -> Series[bool]:
        """Heat index must not exceed 120 °C (physical impossibility otherwise)."""
        return series <= 120.0

    @pa.check("temp_z_score", name="z_score_range")
    @classmethod
    def z_score_realistic(cls, series: Series[float]) -> Series[bool]:
        """Z-scores outside ±10 indicate a data anomaly, not a real measurement."""
        return series.abs() <= 10.0


print("Schema classes defined: RawSchema, CleanSchema, FeatureSchema")
print(f"  RawSchema columns    : {list(RawSchema.to_schema().columns.keys())}")
print(f"  CleanSchema columns  : {list(CleanSchema.to_schema().columns.keys())}")
print(f"  FeatureSchema columns: {list(FeatureSchema.to_schema().columns.keys())}")


**What just happened?**

- **`pa.DataFrameModel`** lets you define schemas as classes — column specs become type-annotated class attributes with `pa.Field` kwargs.
- **`pa.Field(ge=..., le=...)`** replaces `pa.Column(float, pa.Check.ge(...))` — same semantics, more readable.
- **`Config.coerce = True`** enables automatic dtype coercion — strings become floats if possible, reducing boilerplate at the ingestion stage.
- **`@pa.check` classmethods** on `FeatureSchema` add custom vectorized validations that Pandera runs automatically during schema validation.


## Step 2 · Implement the three pipeline stage functions

Each function is decorated with `@check_input` and `@check_output` to enforce contracts at both boundaries. The function bodies contain only the transformation logic — validation is handled by the decorators.


In [ ]:
import pandas as pd
import numpy as np
import pandera as pa
from pandera import check_input, check_output

# ── Stage 1: Raw ingestion ─────────────────────────────────────────────────
def ingest_raw(records: list) -> pd.DataFrame:
    """Convert raw sensor records (list of dicts) into a validated DataFrame."""
    df = pd.DataFrame(records)
    # Validate the freshly constructed raw DataFrame
    return RawSchema.validate(df)


# ── Stage 2: Cleaning ──────────────────────────────────────────────────────
@check_input(RawSchema.to_schema())
@check_output(CleanSchema.to_schema())
def clean_readings(df: pd.DataFrame) -> pd.DataFrame:
    """Drop nulls and clip any remaining out-of-range values."""
    df = df.dropna(subset=["temperature", "humidity"])
    df = df.copy()
    # Clip to safe physical ranges (catches edge cases after null removal)
    df["temperature"] = df["temperature"].clip(-40.0, 85.0)
    df["humidity"]    = df["humidity"].clip(0.0, 100.0)
    return df.reset_index(drop=True)


# ── Stage 3: Feature engineering ──────────────────────────────────────────
@check_input(CleanSchema.to_schema())
@check_output(FeatureSchema.to_schema())
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add heat_index and temp_z_score derived features."""
    df = df.copy()
    # Simplified heat-index formula (Rothfusz subset, safe for normal ranges)
    df["heat_index"] = (
        -8.78469475556
        + 1.61139411 * df["temperature"]
        + 2.33854883889 * df["humidity"]
        - 0.14611605 * df["temperature"] * df["humidity"]
        - 0.012308094 * df["temperature"] ** 2
        - 0.016424828 * df["humidity"] ** 2
        + 0.002211732 * df["temperature"] ** 2 * df["humidity"]
        + 0.00072546 * df["temperature"] * df["humidity"] ** 2
        - 0.000003582 * df["temperature"] ** 2 * df["humidity"] ** 2
    )
    # Z-score of temperature across this batch
    mu, sigma = df["temperature"].mean(), df["temperature"].std()
    df["temp_z_score"] = (df["temperature"] - mu) / sigma if sigma > 0 else 0.0
    return df


print("Pipeline stage functions defined:")
print("  ingest_raw()        → RawSchema")
print("  clean_readings()    → RawSchema → CleanSchema")
print("  engineer_features() → CleanSchema → FeatureSchema")


**What just happened?**

- **`RawSchema.validate(df)`** in `ingest_raw` uses `DataFrameModel` directly — `coerce=True` handles dtype casting.
- **`.to_schema()`** converts a `DataFrameModel` to a `DataFrameSchema` object needed by `check_input` / `check_output`.
- The heat-index formula uses only standard numpy arithmetic — no external API calls, works in a fresh Colab.
- **`sigma > 0`** guard prevents division by zero when all temperatures in a batch are identical.


## Step 3 · Run the happy-path pipeline end-to-end


In [ ]:
import pandas as pd

# Simulate sensor records arriving from the ingestion layer
raw_records = [
    {"device_id": "SEN-001", "temperature": 22.5, "humidity": 55.0, "timestamp": "2024-06-01T10:00:00"},
    {"device_id": "SEN-002", "temperature": 30.1, "humidity": 72.3, "timestamp": "2024-06-01T10:01:00"},
    {"device_id": "SEN-003", "temperature": None, "humidity": 60.0, "timestamp": "2024-06-01T10:02:00"},  # null → dropped
    {"device_id": "SEN-004", "temperature": 18.7, "humidity": None, "timestamp": "2024-06-01T10:03:00"},  # null → dropped
    {"device_id": "SEN-005", "temperature": 26.4, "humidity": 48.9, "timestamp": "2024-06-01T10:04:00"},
]

# Stage 1: Ingest
raw_df = ingest_raw(raw_records)
print(f"Stage 1 — Raw: {len(raw_df)} rows")
print(raw_df[["device_id", "temperature", "humidity"]].to_string())

# Stage 2: Clean
clean_df = clean_readings(raw_df)
print(f"\nStage 2 — Cleaned: {len(clean_df)} rows (2 nulls dropped)")
print(clean_df[["device_id", "temperature", "humidity"]].to_string())

# Stage 3: Feature engineering
feature_df = engineer_features(clean_df)
print(f"\nStage 3 — Features: {len(feature_df)} rows")
print(feature_df[["device_id", "temperature", "heat_index", "temp_z_score"]].to_string())


**What just happened?**

- All three stages ran sequentially; the pipeline started with 5 records and finished with 3 (2 null rows dropped in cleaning).
- Every stage transition was validated by `@check_input` / `@check_output` — no explicit `validate()` calls in the pipeline code itself.
- The `heat_index` and `temp_z_score` columns passed the custom `@pa.check` classmethods in `FeatureSchema`.
- The decorators act as **invisible contracts** — the pipeline logic reads cleanly without validation noise.


## Step 4 · Lazy validation report across all stages

In production you don't want the pipeline to crash on the first error — you want a **complete report** of everything that's wrong so the data engineering team can fix it in one pass. We achieve this by running each stage with `lazy=True` and collecting errors.


In [ ]:
import pandas as pd
import pandera as pa

def run_pipeline_with_report(records: list) -> dict:
    """
    Run the full pipeline with lazy=True at each stage.
    Returns a dict with 'data' (final features) and 'errors' (list of failure DataFrames).
    """
    report = {"errors": [], "data": None}

    # Stage 1: ingest and validate lazily
    df = pd.DataFrame(records)
    try:
        df = RawSchema.to_schema().validate(df, lazy=True)
    except pa.errors.SchemaErrors as errs:
        report["errors"].append(("RawSchema", errs.failure_cases.copy()))
        return report  # can't continue if raw data is fundamentally broken

    # Stage 2: clean (uses decorator; call to_schema().validate directly for lazy mode)
    df = df.dropna(subset=["temperature", "humidity"]).copy()
    df["temperature"] = df["temperature"].clip(-40.0, 85.0)
    df["humidity"]    = df["humidity"].clip(0.0, 100.0)
    df = df.reset_index(drop=True)
    try:
        df = CleanSchema.to_schema().validate(df, lazy=True)
    except pa.errors.SchemaErrors as errs:
        report["errors"].append(("CleanSchema", errs.failure_cases.copy()))

    # Stage 3: feature engineering
    df = df.copy()
    df["heat_index"] = (
        -8.78 + 1.61 * df["temperature"] + 2.33 * df["humidity"]
        - 0.146 * df["temperature"] * df["humidity"]
    ).clip(-50, 120)  # keep bounded
    mu, sigma = df["temperature"].mean(), df["temperature"].std()
    df["temp_z_score"] = (df["temperature"] - mu) / sigma if sigma > 0 else 0.0
    try:
        df = FeatureSchema.to_schema().validate(df, lazy=True)
    except pa.errors.SchemaErrors as errs:
        report["errors"].append(("FeatureSchema", errs.failure_cases.copy()))

    report["data"] = df
    return report


# Run with clean data first
result = run_pipeline_with_report(raw_records)
print(f"Pipeline finished. Errors: {len(result['errors'])}, Output rows: {len(result['data'])}")

# Inject bad data
bad_records = [
    {"device_id": "SEN-001", "temperature": 200.0,  "humidity": 55.0,  "timestamp": "2024-06-01T10:00:00"},  # temp > 85
    {"device_id": "SN",      "temperature": 22.5,   "humidity": -10.0, "timestamp": "2024-06-01T10:01:00"},  # id too short + humidity < 0
    {"device_id": "SEN-003", "temperature": 30.0,   "humidity": 60.0,  "timestamp": "2024-06-01T10:02:00"},
]

bad_result = run_pipeline_with_report(bad_records)
print(f"\nPipeline with bad data. Error stages: {[s for s,_ in bad_result['errors']]}")
for stage, fc in bad_result["errors"]:
    print(f"\n  [{stage}] failures:")
    print(fc[["column", "check", "failure_case", "index"]].to_string())


**What just happened?**

- `run_pipeline_with_report` wraps each stage in a `try/except pa.errors.SchemaErrors` block with `lazy=True`, collecting failures without crashing.
- Bad records produced errors in `RawSchema` — the short `device_id` and out-of-range temperature and humidity values were all captured in one pass.
- The report tuple `(stage_name, failure_cases_df)` gives a structured audit trail suitable for logging to a data quality dashboard.
- The pipeline returns partial results (`data`) when only later stages fail — callers can decide whether to proceed with warnings.


## Step 5 · Unit tests — happy path and sad path for each stage


In [ ]:
import pandas as pd
import pandera as pa

# ── Helpers ────────────────────────────────────────────────────────────────
def make_raw_df(**overrides):
    """Return a minimal valid raw DataFrame with optional column overrides."""
    base = {
        "device_id": ["SEN-001", "SEN-002"],
        "temperature": [22.0, 30.0],
        "humidity": [55.0, 60.0],
        "timestamp": ["2024-06-01T10:00:00", "2024-06-01T10:01:00"],
    }
    base.update(overrides)
    return pd.DataFrame(base)


def make_clean_df(**overrides):
    """Return a minimal valid clean DataFrame."""
    return make_raw_df(**overrides)  # CleanSchema is same structure, no nulls


# ── Stage 1 tests ──────────────────────────────────────────────────────────
def test_ingest_raw_happy():
    df = ingest_raw([
        {"device_id": "SEN-001", "temperature": 22.0, "humidity": 55.0, "timestamp": "2024-06-01T10:00:00"},
    ])
    assert "device_id" in df.columns
    assert len(df) == 1
    print("PASS test_ingest_raw_happy")


def test_ingest_raw_sad_short_id():
    try:
        ingest_raw([{"device_id": "S", "temperature": 22.0, "humidity": 55.0, "timestamp": "2024-06-01T10:00:00"}])
        raise AssertionError("Expected SchemaError")
    except pa.errors.SchemaError:
        print("PASS test_ingest_raw_sad_short_id")


# ── Stage 2 tests ──────────────────────────────────────────────────────────
def test_clean_drops_nulls():
    df = make_raw_df(temperature=[None, 30.0], humidity=[55.0, 60.0])
    result = clean_readings(df)
    assert len(result) == 1
    assert result["temperature"].isna().sum() == 0
    print("PASS test_clean_drops_nulls")


def test_clean_sad_bad_input():
    """Temperature > 85 in the input fails RawSchema check_input."""
    bad = make_raw_df(temperature=[200.0, 30.0])
    try:
        clean_readings(bad)
        raise AssertionError("Expected SchemaError")
    except pa.errors.SchemaError:
        print("PASS test_clean_sad_bad_input")


# ── Stage 3 tests ──────────────────────────────────────────────────────────
def test_engineer_features_happy():
    df = make_clean_df()
    result = engineer_features(df)
    assert "heat_index" in result.columns
    assert "temp_z_score" in result.columns
    assert result["heat_index"].notna().all()
    print("PASS test_engineer_features_happy")


def test_engineer_features_sad_missing_col():
    """Passing a DataFrame without 'humidity' fails CleanSchema check_input."""
    no_humidity = pd.DataFrame({
        "device_id": ["SEN-001"],
        "temperature": [22.0],
        "timestamp": ["2024-06-01T10:00:00"],
    })
    try:
        engineer_features(no_humidity)
        raise AssertionError("Expected SchemaError")
    except pa.errors.SchemaError:
        print("PASS test_engineer_features_sad_missing_col")


# Run all unit tests
test_ingest_raw_happy()
test_ingest_raw_sad_short_id()
test_clean_drops_nulls()
test_clean_sad_bad_input()
test_engineer_features_happy()
test_engineer_features_sad_missing_col()
print("\nAll unit tests passed.")


**What just happened?**

- Each pipeline stage has a **happy-path** test (valid data flows through) and a **sad-path** test (invalid data raises `SchemaError`).
- The `make_raw_df` / `make_clean_df` helpers with `**overrides` make it easy to create one-column mutations without repeating boilerplate.
- Sad-path tests use the same try/except pattern: raise `AssertionError` if the expected `SchemaError` does NOT appear.
- In a real project these would live in `tests/test_pipeline.py` and run via `pytest tests/`.


## Step 6 · Hypothesis property-based test for the full pipeline

Now we verify that **for any valid raw input**, the pipeline produces output that satisfies `FeatureSchema`. This is the most powerful form of testing — it explores the entire valid input space automatically.


In [ ]:
import pandera as pa
import pandas as pd
from hypothesis import given, settings, HealthCheck
import hypothesis.strategies as st

# Build a composite strategy that generates valid RawSchema DataFrames
# (We use a composite strategy since DataFrameModel.strategy() requires pandera[hypotheses])
@st.composite
def raw_df_strategy(draw):
    n = draw(st.integers(min_value=2, max_value=8))  # 2–8 rows
    ids = draw(st.lists(
        st.text(alphabet="ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-", min_size=3, max_size=10),
        min_size=n, max_size=n
    ))
    temps = draw(st.lists(
        st.floats(min_value=-40.0, max_value=85.0, allow_nan=False),
        min_size=n, max_size=n
    ))
    humidities = draw(st.lists(
        st.floats(min_value=0.0, max_value=100.0, allow_nan=False),
        min_size=n, max_size=n
    ))
    timestamps = ["2024-06-01T10:00:00"] * n  # fixed timestamp for simplicity
    return pd.DataFrame({
        "device_id": ids,
        "temperature": temps,
        "humidity": humidities,
        "timestamp": timestamps,
    })


@given(raw_df_strategy())
@settings(
    max_examples=25,
    suppress_health_check=[HealthCheck.too_slow, HealthCheck.filter_too_much],
)
def test_pipeline_end_to_end_invariant(df):
    """
    Property: for any valid raw input, the pipeline must produce
    output that satisfies FeatureSchema.
    """
    try:
        # Run ingest (validate raw)
        raw = RawSchema.to_schema().validate(df)
        # Run clean
        clean = clean_readings(raw)
        if len(clean) == 0:
            return  # empty after null drop — skip (no nulls in strategy, so rare)
        # Run features
        features = engineer_features(clean)
        # Assert FeatureSchema holds
        FeatureSchema.to_schema().validate(features)
    except pa.errors.SchemaError as exc:
        raise AssertionError(f"Pipeline invariant violated: {exc}")


test_pipeline_end_to_end_invariant()
print("Hypothesis property test passed: 25 generated raw inputs all produced valid FeatureSchema output.")


**What just happened?**

- The `@st.composite` strategy generates DataFrames that are **guaranteed** to satisfy `RawSchema` — random but valid.
- **`@given(raw_df_strategy())`** feeds 25 such DataFrames through the entire three-stage pipeline.
- Any `SchemaError` in the feature stage (e.g., from a custom check on `heat_index` or `temp_z_score`) is caught and re-raised as `AssertionError`, which Hypothesis reports with the minimal failing example.
- This test is the **highest confidence check** in our suite: it probes the pipeline with inputs we didn't think of.


## Step 7 · Custom `@pa.check` classmethods in detail

The two custom checks on `FeatureSchema` (`heat_index_upper_bound` and `z_score_realistic`) are worth examining closely — they illustrate how to add **domain-specific rules** that go beyond built-in `pa.Field` constraints.


In [ ]:
import pandas as pd
import pandera as pa

# Demonstrate the custom classmethods firing on deliberately bad feature output

# Construct a DataFrame that would pass CleanSchema but whose features
# would violate the custom checks if we injected bad feature values
malicious_features = pd.DataFrame({
    "device_id": ["SEN-001"],
    "temperature": [22.0],
    "humidity": [55.0],
    "timestamp": ["2024-06-01T10:00:00"],
    "heat_index": [999.0],   # violates heat_index_upper_bound (>120)
    "temp_z_score": [0.5],
})

try:
    FeatureSchema.to_schema().validate(malicious_features, lazy=True)
except pa.errors.SchemaErrors as errs:
    print("Custom classmethod checks caught:")
    print(errs.failure_cases[["column", "check", "failure_case"]].to_string())

# Now test z_score_realistic
bad_zscore = pd.DataFrame({
    "device_id": ["SEN-001"],
    "temperature": [22.0],
    "humidity": [55.0],
    "timestamp": ["2024-06-01T10:00:00"],
    "heat_index": [25.0],
    "temp_z_score": [99.0],  # |99| > 10 → anomaly check fires
})

try:
    FeatureSchema.to_schema().validate(bad_zscore, lazy=True)
except pa.errors.SchemaErrors as errs:
    print("\nZ-score anomaly check caught:")
    print(errs.failure_cases[["column", "check", "failure_case"]].to_string())


**What just happened?**

- **`@pa.check("heat_index", name="heat_index_bounded")`** registers a classmethod as a custom vectorized check on that column.
- The classmethod receives a `pd.Series` and must return a boolean `Series` — `True` = valid, `False` = violation.
- Named checks (`name=`) appear in `failure_cases["check"]` — crucial for readable error reports.
- **Both custom checks fire independently** in `lazy=True` mode — all violations visible in one pass.


In [ ]:
# Challenge: Extend the pipeline with a fourth stage
#
# 1. Define AlertSchema(pa.DataFrameModel) with columns:
#    - device_id: str
#    - heat_index: float
#    - alert_level: str, must be one of ["green", "yellow", "red"]
#    Add a custom @pa.check classmethod that asserts:
#    when heat_index >= 38.0, alert_level must NOT be "green"
#    (Hint: return a bool Series where the condition is satisfied)
#
# 2. Write classify_alerts(df) decorated with @check_input(FeatureSchema.to_schema())
#    and @check_output(AlertSchema.to_schema()):
#    - Maps heat_index < 27 → "green", 27–38 → "yellow", >= 38 → "red"
#
# 3. Write one happy-path and one sad-path unit test.
#
# 4. (Bonus) Write a @given property test that feeds classify_alerts
#    with schema.strategy()-generated FeatureSchema DataFrames.

# Your solution here


---
## Day 7 key concepts recap

| Concept | What to remember |
|---|---|
| `pa.DataFrameModel` | Class-based schema; use `pa.Field` kwargs for constraints |
| `.to_schema()` | Convert `DataFrameModel` → `DataFrameSchema` for decorator use |
| `@pa.check` classmethod | Custom vectorized check on a named column; returns bool Series |
| Custom check `name=` | Appears in `failure_cases["check"]` for readable reports |
| `@check_input` + `@check_output` | Guard each pipeline stage at both boundaries |
| Lazy validation report | `try: validate(lazy=True) except SchemaErrors` pattern per stage |
| Unit tests | Happy + sad path per stage; use `make_*_df(**overrides)` helpers |
| `@given` property test | `@st.composite` strategy → feeds 25+ random inputs through the full pipeline |
| Pipeline pattern | `ingest_raw` → `clean_readings` → `engineer_features`; schemas are contracts |

> **Tip:** In production, store each stage's `failure_cases` DataFrame to a validation log table (BigQuery, Snowflake, DuckDB) for trend analysis. Pandera makes this trivial — `failure_cases` is already a DataFrame.

---
## What's next
**Course complete!** You've covered the full Pandera toolkit: DataFrameSchema, DataFrameModel, multi-backend validation, function decorators, Hypothesis integration, schema inference, lazy reporting, and capstone pipeline design.

Next steps:
- Explore **FastAPI + Pandera** for HTTP request/response validation.
- Integrate `pandera` checks into an **Apache Airflow** or **Prefect** task.
- Try **pandera.polars** in a Rust-backed production pipeline for maximum throughput.

Mark Day 7 complete in your [tracker](../index.html).
